# Índice vectorial en GPU — CATSNebula

Corre `encode_index.py` sobre `chunks.jsonl` en una GPU de Colab y baja
`index.faiss` + `metadata.jsonl` + `manifiesto.json`.

**Por qué la nube:** medido en el CPU del equipo (Ryzen 5 5600G, 6 núcleos,
fp32), BGE-M3 tarda **6,9 s/chunk** → ~164 h para los 86.046 chunks. En GPU
son entre 1 y 4 h según la tarjeta (fp32 sin TF32; una T4 es lo más lento del
rango, una L4/A100 lo más rápido). La celda de muestra mide antes de
comprometer las horas.

**Antes de empezar:**
1. `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU`.
2. Subir `data/chunks.jsonl` (234 MB) a Drive en `MyDrive/catsnebula/chunks.jsonl`.
   No está en git: pesa más que el límite de 100 MB por archivo de GitHub.

**Sobre el determinismo:** los vectores de GPU no son bit-idénticos a los de
CPU. El determinismo que pide la fase es interno (poder rehacer la corrida),
no algo que valide el evaluador — los organizadores confirmaron que cargan
nuestro índice, no lo reconstruyen. `manifiesto.json` deja registrado el
dispositivo exacto, la revisión del modelo y las versiones, que es lo que
hace falta para reproducirla. Por eso **fp32 sin TF32**: mantiene el índice
comparable entre corridas en la misma tarjeta.

In [ ]:
# 1. Confirmar que hay GPU (si esto sale vacío, falta cambiar el entorno)
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# 2. Dependencias. Mismas versiones que requirements.txt salvo torch, que
#    viene con Colab compilado contra su driver CUDA — pinnearlo aquí suele
#    romper más de lo que arregla. El manifiesto registra cuál se usó.
!pip install -q sentence-transformers==5.7.0 transformers==5.15.0 faiss-cpu==1.15.0
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# 3. Código del repo
!git clone -q https://github.com/Juancabel/AdAstra2026-CATSNebula.git /content/repo
%cd /content/repo
!git log --oneline -1

In [ ]:
# 4. Drive: de ahí sale chunks.jsonl y ahí van los parciales.
#
#    La salida va a Drive a propósito: Colab corta sesiones, y encode_index.py
#    vuelca el parcial cada 50 batches. Si se corta, se reanuda con --reanudar
#    desde el último batch completo en vez de volver a empezar.
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')

CHUNKS = Path('/content/drive/MyDrive/catsnebula/chunks.jsonl')
SALIDA = Path('/content/drive/MyDrive/catsnebula/encoder_bge-m3')
assert CHUNKS.exists(), f'Falta {CHUNKS}: subí data/chunks.jsonl a esa ruta.'
print(f'{CHUNKS.stat().st_size / 1e6:.0f} MB')
print(sum(1 for _ in CHUNKS.open('rb')), 'chunks')  # esperado: 86046

In [ ]:
# 5. Muestra de validación: mide ms/chunk en ESTA GPU y proyecta el total.
#    Descarga BGE-M3 (~2,3 GB) la primera vez.
!python encode_index.py "{CHUNKS}" /content/muestra --muestra 500 --dispositivo cuda

In [ ]:
# 6. Mapeo 1:1 sobre la muestra, en un proceso nuevo. Si esto falla, NO seguir:
#    un índice desalineado recupera con puntajes razonables y devuelve el texto
#    equivocado, y ninguna métrica de recall lo delata.
!python scripts/verificar_indice.py /content/muestra --n 30 --dispositivo cuda

In [ ]:
# 7. Corrida completa. Mirar antes la proyección de la celda 5.
#    Si la sesión se corta, volver a correr ESTA celda con --reanudar añadido.
!python encode_index.py "{CHUNKS}" "{SALIDA}" --dispositivo cuda

In [ ]:
# 8. Verificación final sobre el índice completo
!python scripts/verificar_indice.py "{SALIDA}" --n 50 --dispositivo cuda

In [ ]:
# 9. Entrega. Los artefactos ya están en Drive; esto solo los resume.
#    manifiesto.json SÍ se versiona en git: es la trazabilidad de qué corrida
#    produjo la entrega. index.faiss y metadata.jsonl no (pesan de más).
import json
for f in sorted(SALIDA.iterdir()):
    print(f'{f.stat().st_size / 1e6:9.1f} MB  {f.name}')
print()
print(json.dumps(json.loads((SALIDA / 'manifiesto.json').read_text()), indent=2, ensure_ascii=False))